In [ ]:
!pip install -q transformers accelerate torch fastapi uvicorn pyngrok python-multipart nest-asyncio

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading Qwen model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
)

model.to("cuda")
model.eval()

print("Model loaded successfully on GPU")

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import json, re

app = FastAPI()

from fastapi.middleware.cors import CORSMiddleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173", "http://localhost:5174", "http://127.0.0.1:5173"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class SchemaRequest(BaseModel):
    extractedText: str

class ProfileRequest(BaseModel):
    resumeText: str

class AutofillRequest(BaseModel):
    profile: dict
    formSchema: list   # stays a LIST, matching your existing flat array format

print("FastAPI initialized")

In [ ]:
FORM_PROMPT = """
You are an expert AI system that converts OCR text from a printed form into a structured dynamic web form schema.

The OCR text may contain:
- form titles
- institution names
- section headings
- field labels
- blank lines
- underscores
- checkboxes
- radio buttons
- options
- instructions
- declarations
- signatures
- tables
- page numbers
- noisy OCR text

Your job is to identify the actual USER-INPUT fields and the document structure.

IMPORTANT:
Do NOT treat every line as a heading.

==================================================
1. FORM TITLE / MAIN HEADING
==================================================

Use:

{
  "type": "heading",
  "label": "..."
}

ONLY for:
- document title
- institution name
- major title

Examples:

TRANSFER CERTIFICATE APPLICATION FORM

APPLICATION FOR ISSUE OF BONAFIDE CERTIFICATE

SREE VENKATA INSTITUTE OF TECHNOLOGY

These do NOT require user input.

==================================================
2. SECTION
==================================================

Use:

{
  "type": "section",
  "label": "..."
}

for logical groups such as:

STUDENT DETAILS
APPLICANT DETAILS
PARENT DETAILS
ACADEMIC INFORMATION
CONTACT INFORMATION
DOCUMENT DETAILS
DECLARATION

Do NOT create a text input for these.

==================================================
3. ACTUAL INPUT FIELD
==================================================

If the user is expected to enter information, create an input field.

Examples:

Application No.
Application Number
Student Name
Name of Student
Register Number
Date of Birth
Mobile Number
Email Address
Father's Name
Mother's Name
Address
Academic Year

These are NOT headings.

==================================================
4. FIELD TYPES
==================================================

Use exactly one of these types:

heading
section
text
textarea
number
date
email
tel
radio
checkbox
select
file

Rules:

Name / Student Name / Father's Name / Mother's Name
-> text

Application Number / Register Number / ID
-> text

Date / Date of Birth / DOB
-> date

Email / Email Address
-> email

Mobile / Phone / Telephone
-> tel

Age / Marks / Percentage / Number
-> number

Address / Permanent Address / Communication Address
-> textarea

Long description / Remarks
-> textarea

Gender with choices
-> radio

A single-choice list / dropdown
-> select

Multiple selectable choices
-> checkbox

Document upload
-> file

==================================================
5. OPTIONS
==================================================

Only include "options" for:

radio
select
checkbox

Example:

Gender:
Male [ ]
Female [ ]
Other [ ]

becomes:

{
  "type": "radio",
  "name": "gender",
  "label": "Gender",
  "required": false,
  "options": [
    "Male",
    "Female",
    "Other"
  ]
}

IMPORTANT:
Do NOT invent options.

If options are not visible in the OCR, do not invent them.

The Python post-processing layer will handle missing options using a controlled domain dictionary when appropriate.

Therefore:
- If options are visible in the OCR, extract them.
- If options are NOT visible, return an empty or missing "options" field.
- Never invent domain options yourself.
- Do not guess options based only on general knowledge.

==================================================
6. REQUIRED
==================================================

Use:

"required": true

ONLY if the form clearly indicates the field is mandatory.

Examples:

Name *
Email *
Phone (Required)

Otherwise:

"required": false

==================================================
7. NAME
==================================================

Every actual input field MUST have a unique machine-readable name.

Convert the label to camelCase.

Examples:

Application No.
-> applicationNo

Student Name
-> studentName

Date of Birth
-> dateOfBirth

Mobile Number
-> mobileNumber

Email Address
-> emailAddress

Father's Name
-> fathersName

Aadhaar / ID Number
-> aadhaarIdNumber

==================================================
8. VERY IMPORTANT CLASSIFICATION RULE
==================================================

Look at the surrounding OCR text.

If a label is followed by:
- underscores
- blank spaces
- colon
- empty area
- checkbox
- radio option
- input-like area

then it is probably an INPUT FIELD.

Example:

Application No. ____________

MUST become:

{
  "type": "text",
  "name": "applicationNo",
  "label": "Application No.",
  "required": false
}

NOT:

{
  "type": "heading",
  "label": "Application No."
}

==================================================
9. DO NOT INVENT
==================================================

Do NOT invent:
- fields
- options
- values
- dates
- names
- addresses
- validations

Only use information supported by the OCR.

The Python post-processing layer may add controlled domain fallback options ONLY when:
1. the field is a select/radio/checkbox field,
2. the document did not provide options,
3. the field label exactly matches a supported domain key.

The Python layer will also calculate:
- optionsSource
- organization/address header information
- statistics

Do NOT generate these metadata fields yourself.

==================================================
10. REMOVE OCR NOISE
==================================================

Ignore meaningless OCR artifacts such as:

page numbers
isolated symbols
random punctuation
empty lines
repeated separators
unreadable fragments

Correct obvious OCR spacing problems when the meaning is clear.

==================================================
11. DUPLICATES
==================================================

Do not generate duplicate fields.

If the same field appears repeatedly because of OCR duplication, keep only the meaningful occurrence.

==================================================
12. OUTPUT FORMAT
==================================================

Return ONLY valid JSON.

The output MUST be a flat JSON array.

Example:

[
  {
    "type": "heading",
    "label": "TRANSFER CERTIFICATE APPLICATION FORM"
  },
  {
    "type": "section",
    "label": "STUDENT DETAILS"
  },
  {
    "type": "text",
    "name": "studentName",
    "label": "Student Name",
    "required": true
  },
  {
    "type": "date",
    "name": "dateOfBirth",
    "label": "Date of Birth",
    "required": false
  }
]

DO NOT return:
- markdown
- ```json
- explanations
- comments
- additional metadata
- header objects
- statistics
- optionsSource
- formId
- nested elements

Return ONLY the flat JSON array of form elements.

==================================================
13. FINAL QUALITY CHECK
==================================================

Before returning the answer, verify:

1. Every actual input field has:
   - type
   - name
   - label
   - required

2. Every name is unique.

3. Headings do NOT have input names.

4. Sections do NOT have input names.

5. Only radio/select/checkbox fields may contain options.

6. Options are included only when supported by the OCR.

7. No fields are invented.

8. No options are invented.

9. OCR noise is removed.

10. The response is valid JSON.

11. The response is a flat JSON array.

12. Return nothing except the JSON array.

==================================================
OCR TEXT:
"""



In [ ]:

# ============================================================
# JSON NORMALIZATION + VALIDATION
# ============================================================

ALLOWED_TYPES = {
    "heading",
    "section",
    "text",
    "textarea",
    "number",
    "date",
    "email",
    "tel",
    "radio",
    "checkbox",
    "select",
    "file"
}


TYPE_ALIASES = {
    "input": "text",
    "textfield": "text",
    "textinput": "text",
    "string": "text",

    "phone": "tel",
    "mobile": "tel",
    "telephone": "tel",

    "birthdate": "date",
    "dob": "date",
    "datetime": "date",

    "longtext": "textarea",
    "multiline": "textarea",

    "dropdown": "select",
    "choice": "select",

    "options": "select",

    "boolean": "checkbox",
    "check": "checkbox",

    "title": "heading",

    "subheading": "section"
}


# ============================================================
# DOMAIN FALLBACK OPTIONS
# ============================================================
# Used ONLY when the document does not provide options.
# Never override options extracted from the document.

DOMAIN_OPTIONS = {
    "gender": ["Male", "Female", "Other"],
    "sex": ["Male", "Female", "Other"],
    "bloodgroup": [
        "A+", "A-", "B+", "B-",
        "AB+", "AB-", "O+", "O-"
    ],
    "degree": [
        "B.E", "B.Tech",
        "M.E", "M.Tech",
        "MBA", "MCA", "PhD"
    ],
}


def domain_key(label):
    """
    Convert a label into a normalized domain key.

    Examples:
        Gender       -> gender
        Blood Group  -> bloodgroup
        Blood-Group  -> bloodgroup
    """

    return re.sub(
        r"[^a-z]",
        "",
        clean_text(label).lower()
    )


def apply_options_source(field):
    """
    Apply optionsSource to choice-based fields.

    Possible values:

        document
        domain
        none

    Rules:

    1. If options came from OCR/document:
       optionsSource = document

    2. If no document options exist but a controlled
       domain fallback exists:
       optionsSource = domain

    3. Otherwise:
       optionsSource = none
    """

    if field.get("type") not in {
        "select",
        "radio",
        "checkbox"
    }:
        return field

    # Options extracted from the document
    if field.get("options"):
        field["optionsSource"] = "document"
        return field

    # Try controlled domain fallback
    key = domain_key(
        field.get("label", "")
    )

    if key in DOMAIN_OPTIONS:

        field["options"] = DOMAIN_OPTIONS[key]
        field["optionsSource"] = "domain"

    else:

        field["options"] = field.get(
            "options",
            []
        )

        field["optionsSource"] = "none"

    return field


# ============================================================
# TEXT CLEANING
# ============================================================

def clean_text(value):

    if value is None:
        return ""

    value = str(value)

    value = value.replace("\n", " ")
    value = value.replace("\r", " ")

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


# ============================================================
# FIELD NAME GENERATION
# ============================================================

def make_field_name(label):
    """
    Convert:

        Student Name
            -> studentName

        Date of Birth
            -> dateOfBirth

        Application No.
            -> applicationNo
    """

    label = clean_text(label)

    label = label.lower()

    replacements = {

        "application no": "application number",
        "app no": "application number",

        "reg no": "register number",
        "register no": "register number",

        "dob": "date of birth",

        "mobile no": "mobile number",
        "phone no": "phone number",

        "email id": "email address",

        "aadhaar no": "aadhaar number"
    }

    for old, new in replacements.items():
        label = label.replace(
            old,
            new
        )

    # Remove punctuation
    label = re.sub(
        r"[^a-zA-Z0-9\s]",
        " ",
        label
    )

    words = label.split()

    if not words:
        return "field"

    name = words[0]

    for word in words[1:]:

        if word:
            name += (
                word[0].upper()
                + word[1:]
            )

    return name


# ============================================================
# TYPE NORMALIZATION
# ============================================================

def normalize_type(
    field_type,
    label=""
):

    if not field_type:
        field_type = ""

    field_type = str(
        field_type
    ).lower().strip()

    field_type = field_type.replace(
        "-",
        ""
    )

    field_type = field_type.replace(
        "_",
        ""
    )

    field_type = field_type.replace(
        " ",
        ""
    )

    if field_type in TYPE_ALIASES:
        return TYPE_ALIASES[field_type]

    if field_type in ALLOWED_TYPES:
        return field_type

    # Infer from label if LLM produced
    # an unknown type

    label_lower = label.lower()

    if "email" in label_lower:
        return "email"

    if any(
        x in label_lower
        for x in [
            "mobile",
            "phone",
            "telephone"
        ]
    ):
        return "tel"

    if any(
        x in label_lower
        for x in [
            "date",
            "dob",
            "birth"
        ]
    ):
        return "date"

    if any(
        x in label_lower
        for x in [
            "address",
            "remarks",
            "description"
        ]
    ):
        return "textarea"

    if any(
        x in label_lower
        for x in [
            "gender",
            "sex"
        ]
    ):
        return "radio"

    return "text"


# ============================================================
# OPTION CLEANING
# ============================================================

def clean_options(options):

    if not isinstance(
        options,
        list
    ):
        return []

    result = []

    for option in options:

        option = clean_text(
            option
        )

        if not option:
            continue

        if option not in result:
            result.append(option)

    return result


# ============================================================
# FIELD NORMALIZATION
# ============================================================

def normalize_field(
    field,
    used_names
):

    if not isinstance(
        field,
        dict
    ):
        return None

    label = clean_text(
        field.get("label")
        or field.get("name")
        or ""
    )

    if not label:
        return None

    raw_type = field.get(
        "type",
        "text"
    )

    field_type = normalize_type(
        raw_type,
        label
    )

    # ========================================================
    # HEADING
    # ========================================================

    if field_type == "heading":

        return {
            "type": "heading",
            "label": label
        }

    # ========================================================
    # SECTION
    # ========================================================

    if field_type == "section":

        return {
            "type": "section",
            "label": label
        }

    # ========================================================
    # ACTUAL INPUT FIELD
    # ========================================================

    name = clean_text(
        field.get("name")
    )

    if not name:
        name = make_field_name(
            label
        )

    # Ensure unique name
    original_name = name
    counter = 2

    while name in used_names:

        name = (
            f"{original_name}{counter}"
        )

        counter += 1

    used_names.add(name)

    # ========================================================
    # REQUIRED
    # ========================================================

    required = field.get(
        "required",
        False
    )

    if isinstance(
        required,
        str
    ):

        required = (
            required.lower()
            in {
                "true",
                "yes",
                "required",
                "mandatory"
            }
        )

    if not isinstance(
        required,
        bool
    ):
        required = False

    # ========================================================
    # STANDARD RESULT
    # ========================================================

    result = {

        "type": field_type,

        "name": name,

        "label": label,

        "required": required
    }

    # ========================================================
    # OPTIONS
    # ========================================================

    if field_type in {
        "radio",
        "select",
        "checkbox"
    }:

        options = clean_options(
            field.get(
                "options",
                []
            )
        )

        if options:
            result["options"] = options

    return result


# ============================================================
# FORM SCHEMA NORMALIZATION
# ============================================================

def normalize_form_schema(
    raw_schema
):
    """
    Converts messy LLM JSON into our
    application's standard schema.
    """

    if not isinstance(
        raw_schema,
        list
    ):
        return []

    normalized = []

    used_names = set()

    for field in raw_schema:

        cleaned = normalize_field(
            field,
            used_names
        )

        if cleaned is not None:
            normalized.append(
                cleaned
            )

    return normalized


# ============================================================
# HEADER EXTRACTION
# ============================================================

ORG_KEYWORDS = [
    "university",
    "institute",
    "college",
    "school",
    "polytechnic"
]


def extract_header(
    schema,
    ocr_text
):
    """
    Deterministic header extraction.

    Looks at the first few OCR lines for an
    organization name and possible address.

    The first heading is NOT removed from
    the schema.
    """

    header = {
        "organization": "",
        "address": ""
    }

    lines = [
        clean_text(line)
        for line in ocr_text.split("\n")
        if clean_text(line)
    ]

    for index, line in enumerate(
        lines[:4]
    ):

        lower = line.lower()

        if any(
            keyword in lower
            for keyword in ORG_KEYWORDS
        ):

            header[
                "organization"
            ] = line

            if index + 1 < len(lines):

                next_line = lines[
                    index + 1
                ]

                if not any(
                    keyword
                    in next_line.lower()
                    for keyword in ORG_KEYWORDS
                ):

                    # Address/city lines are
                    # generally shorter.
                    if len(next_line) < 60:

                        header[
                            "address"
                        ] = next_line

            break

    return header


# ============================================================
# STATISTICS
# ============================================================

def compute_statistics(
    schema
):
    """
    Calculate form statistics.

    Headings and sections are excluded
    from input-field counts.
    """

    stats = {

        "totalInputFields": 0,

        "headings": 0,

        "sections": 0,

        "requiredFields": 0,

        "typeBreakdown": {}
    }

    for field in schema:

        ftype = field.get(
            "type"
        )

        # Heading
        if ftype == "heading":

            stats[
                "headings"
            ] += 1

            continue

        # Section
        if ftype == "section":

            stats[
                "sections"
            ] += 1

            continue

        # Actual input field
        stats[
            "totalInputFields"
        ] += 1

        stats[
            "typeBreakdown"
        ][ftype] = (
            stats[
                "typeBreakdown"
            ].get(
                ftype,
                0
            ) + 1
        )

        if field.get(
            "required"
        ):

            stats[
                "requiredFields"
            ] += 1

    return stats


# ============================================================
# VALIDATION
# ============================================================

def validate_form_schema(
    schema
):
    """
    Deterministic validation.
    """

    errors = []

    if not isinstance(
        schema,
        list
    ):

        errors.append(
            "Schema must be a list."
        )

        return errors

    names = set()

    for index, field in enumerate(
        schema
    ):

        if not isinstance(
            field,
            dict
        ):

            errors.append(
                f"Field {index} is not an object."
            )

            continue

        field_type = field.get(
            "type"
        )

        label = field.get(
            "label"
        )

        # ====================================================
        # TYPE
        # ====================================================

        if not field_type:

            errors.append(
                f"Field {index} has no type."
            )

        elif field_type not in ALLOWED_TYPES:

            errors.append(
                f"Field {index} has invalid type: "
                f"{field_type}"
            )

        # ====================================================
        # LABEL
        # ====================================================

        if not label:

            errors.append(
                f"Field {index} has no label."
            )

        # ====================================================
        # INPUT NAME
        # ====================================================

        if field_type not in {
            "heading",
            "section"
        }:

            name = field.get(
                "name"
            )

            if not name:

                errors.append(
                    f"Field {index} has no name."
                )

            elif name in names:

                errors.append(
                    f"Duplicate field name: {name}"
                )

            else:

                names.add(name)

        # ====================================================
        # CHOICE FIELD OPTIONS
        # ====================================================

        if field_type in {
            "radio",
            "select",
            "checkbox"
        }:

            options = field.get(
                "options"
            )

            if not options:

                errors.append(
                    f"Field {index} ({label}) "
                    f"has type '{field_type}' "
                    f"but no options "
                    f"(optionsSource="
                    f"{field.get('optionsSource', 'missing')})."
                )

    return errors


print(
    "JSON normalization, header extraction, "
    "statistics, options tagging, and validation "
    "functions loaded."
)



In [ ]:
# ============================================================
# SEMANTIC FIELD CORRECTION
# ============================================================

HEADING_WORDS = {
    "details",
    "information",
    "particulars",
    "declaration",
    "instructions",
    "documents",
    "academic details",
    "student details",
    "applicant details",
    "parent details",
    "personal details",
    "contact details"
}


def looks_like_heading(label):
    """
    Determine whether a label is more likely
    to be a section heading.
    """

    normalized = clean_text(label).lower()

    if not normalized:
        return False

    # Strong section indicators
    if normalized in HEADING_WORDS:
        return True

    # Common section endings
    if normalized.endswith(" details"):
        return True

    if normalized.endswith(" information"):
        return True

    if normalized.endswith(" particulars"):
        return True

    # Major document titles
    title_words = [
        "application form",
        "admission form",
        "certificate",
        "transfer certificate",
        "bonafide certificate",
        "registration form"
    ]

    for word in title_words:
        if word in normalized:
            return True

    return False


def correct_semantics(schema):
    """
    Apply deterministic corrections after LLM generation.
    """

    corrected = []

    for field in schema:

        if not isinstance(field, dict):
            continue

        label = clean_text(
            field.get("label", "")
        )

        if not label:
            continue

        field_type = field.get("type", "text")

        # ------------------------------------------------
        # Never convert clear input labels into headings
        # ------------------------------------------------

        input_keywords = [
            "name",
            "number",
            "no.",
            "email",
            "phone",
            "mobile",
            "address",
            "date",
            "dob",
            "gender",
            "age",
            "course",
            "department",
            "class",
            "year",
            "signature",
            "aadhaar",
            "id"
        ]

        label_lower = label.lower()

        looks_like_input = any(
            keyword in label_lower
            for keyword in input_keywords
        )

        # If LLM incorrectly called a clear field heading,
        # convert it into an input.
        if field_type == "heading" and looks_like_input:

            # But don't change actual document titles
            if not looks_like_heading(label):

                new_field = {
                    "type": normalize_type(
                        "",
                        label
                    ),
                    "name": make_field_name(label),
                    "label": label,
                    "required": False
                }

                # Gender
                if "gender" in label_lower or "sex" in label_lower:
                    new_field["type"] = "radio"

                corrected.append(new_field)
                continue

        corrected.append(field)

    return normalize_form_schema(corrected)


print("Semantic correction loaded.")

In [ ]:

# ============================================================
# RESUME PROFILE EXTRACTION PROMPT
# ============================================================

RESUME_PROMPT = """
You are extracting structured profile information from resume text.

Your task is to extract ONLY information explicitly present in the resume.

IMPORTANT:
- Do NOT invent information.
- Do NOT infer missing information.
- If a value is not found, return an empty string "".
- If skills are not found, return an empty array [].
- Return ONLY a valid JSON object.
- Do NOT return markdown.
- Do NOT return ```json.
- Do NOT include explanations.
- Do NOT include additional keys.

==================================================
REQUIRED OUTPUT
==================================================

Return exactly these keys:

{
  "fullName": "",
  "email": "",
  "phone": "",
  "linkedin": "",
  "github": "",
  "portfolio": "",
  "location": "",
  "skills": []
}

==================================================
EXTRACTION RULES
==================================================

fullName:
- Extract the person's full name.
- Do not use section headings as the name.

email:
- Extract the email address exactly as written.

phone:
- Extract the phone/mobile number exactly as written.

linkedin:
- Extract the LinkedIn profile URL or identifier if present.

github:
- Extract the GitHub profile URL or identifier if present.

portfolio:
- Extract the personal portfolio/personal website URL if present.

location:
- Extract the person's stated city, state, country, or location.

skills:
- Extract technical/professional skills explicitly listed in the resume.
- Return skills as a JSON array of strings.
- Do not invent skills.
- Avoid duplicate skills.

==================================================
DO NOT USE SECTION HEADINGS AS VALUES
==================================================

Do NOT treat headings such as:

OBJECTIVE
SUMMARY
PROFILE
EDUCATION
EXPERIENCE
SKILLS
PROJECTS
CERTIFICATIONS
ACHIEVEMENTS
CONTACT
DECLARATION

as values for any profile field.

==================================================
OUTPUT FORMAT
==================================================

Return ONLY the JSON object.

Example:

{
  "fullName": "John Doe",
  "email": "john@example.com",
  "phone": "+91 9876543210",
  "linkedin": "https://linkedin.com/in/johndoe",
  "github": "https://github.com/johndoe",
  "portfolio": "",
  "location": "Chennai, Tamil Nadu",
  "skills": [
    "Java",
    "Spring Boot",
    "React",
    "MongoDB"
  ]
}

==================================================
RESUME TEXT:
"""

In [ ]:

# ============================================================
# GENERATE FORM SCHEMA API
# ============================================================

@app.post("/generate-schema")
async def generate_schema(req: SchemaRequest):

    ocr_text = req.extractedText.strip()

    if not ocr_text:
        return {
            "error": "OCR text is empty",
            "formSchema": [],
            "elements": [],
            "stats": {
                "totalInputFields": 0,
                "headings": 0,
                "sections": 0,
                "requiredFields": 0,
                "typeBreakdown": {}
            },
            "instructions": [],
            "declarations": []
        }

    print("=" * 70)
    print("GENERATING FORM SCHEMA")
    print("OCR characters:", len(ocr_text))
    print("=" * 70)

    # -------------------------------------------------------
    # STEP 1: LLM GENERATION
    # -------------------------------------------------------

    prompt = FORM_PROMPT + "\n" + ocr_text

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    raw = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print("\nRAW LLM OUTPUT:")
    print(raw[:5000])

    # -------------------------------------------------------
    # STEP 2: REMOVE MARKDOWN
    # -------------------------------------------------------

    cleaned = re.sub(
        r"```json|```",
        "",
        raw,
        flags=re.IGNORECASE
    ).strip()

    # -------------------------------------------------------
    # STEP 3: EXTRACT JSON ARRAY
    # -------------------------------------------------------

    start = cleaned.find("[")
    end = cleaned.rfind("]")

    if start == -1 or end == -1:

        print("ERROR: JSON array not found")

        return {
            "error": "Model did not return a JSON array",
            "raw": raw,
            "formSchema": [],
            "elements": [],
            "stats": {
                "totalInputFields": 0,
                "headings": 0,
                "sections": 0,
                "requiredFields": 0,
                "typeBreakdown": {}
            },
            "instructions": [],
            "declarations": []
        }

    cleaned = cleaned[start:end + 1]

    # -------------------------------------------------------
    # STEP 4: PARSE JSON
    # -------------------------------------------------------

    try:

        parsed = json.loads(cleaned)

    except json.JSONDecodeError as e:

        print("JSON PARSE ERROR:", e)

        return {
            "error": "Model output was not valid JSON",
            "raw": raw,
            "formSchema": [],
            "elements": [],
            "stats": {
                "totalInputFields": 0,
                "headings": 0,
                "sections": 0,
                "requiredFields": 0,
                "typeBreakdown": {}
            },
            "instructions": [],
            "declarations": []
        }

    if not isinstance(parsed, list):

        return {
            "error": "Model did not return a JSON array",
            "raw": cleaned,
            "formSchema": [],
            "elements": [],
            "stats": {
                "totalInputFields": 0,
                "headings": 0,
                "sections": 0,
                "requiredFields": 0,
                "typeBreakdown": {}
            },
            "instructions": [],
            "declarations": []
        }

    print("\nRAW FIELD COUNT:", len(parsed))

    # -------------------------------------------------------
    # STEP 5: NORMALIZE
    # -------------------------------------------------------

    normalized = normalize_form_schema(parsed)

    print(
        "NORMALIZED FIELD COUNT:",
        len(normalized)
    )

    # -------------------------------------------------------
    # STEP 6: SEMANTIC CORRECTION
    # -------------------------------------------------------

    corrected = correct_semantics(normalized)

    print(
        "CORRECTED FIELD COUNT:",
        len(corrected)
    )

    # -------------------------------------------------------
    # STEP 7: APPLY OPTIONS SOURCE
    # -------------------------------------------------------
    #
    # For radio/select/checkbox:
    #
    # 1. Document options -> optionsSource="document"
    # 2. Known domain -> optionsSource="domain"
    # 3. No available options -> optionsSource="none"
    #
    # This prevents Qwen from inventing options.
    # -------------------------------------------------------

    corrected = [
        apply_options_source(field)
        for field in corrected
    ]

    print("\nOPTIONS SOURCE APPLIED")

    # -------------------------------------------------------
    # STEP 8: EXTRACT HEADER
    # -------------------------------------------------------

    header = extract_header(
        corrected,
        ocr_text
    )

    print("\nHEADER:")
    print(
        json.dumps(
            header,
            indent=2,
            ensure_ascii=False
        )
    )

    # -------------------------------------------------------
    # STEP 9: EXTRACT TITLE
    # -------------------------------------------------------

    title = ""

    for field in corrected:

        if field.get("type") == "heading":

            title = clean_text(
                field.get("label", "")
            )

            if title:
                break

    # Fallback:
    # If Qwen did not produce a heading, use the
    # first meaningful OCR line.
    if not title:

        lines = [
            clean_text(line)
            for line in ocr_text.splitlines()
            if clean_text(line)
        ]

        if lines:
            title = lines[0]

    print("\nFORM TITLE:", title)

    # -------------------------------------------------------
    # STEP 10: VALIDATE
    # -------------------------------------------------------

    validation_errors = validate_form_schema(
        corrected
    )

    if validation_errors:

        print("\nVALIDATION ERRORS:")

        for error in validation_errors:
            print("-", error)

    else:

        print("\nSCHEMA VALIDATION: PASSED")

    # -------------------------------------------------------
    # STEP 11: COMPUTE STATISTICS
    # -------------------------------------------------------

    stats = compute_statistics(
        corrected
    )

    print("\nFORM STATISTICS:")

    print(
        json.dumps(
            stats,
            indent=2,
            ensure_ascii=False
        )
    )

    # -------------------------------------------------------
    # STEP 12: FINAL JSON
    # -------------------------------------------------------

    print("\nFINAL FORM SCHEMA:")

    print(
        json.dumps(
            corrected,
            indent=2,
            ensure_ascii=False
        )
    )

    print("=" * 70)

    # -------------------------------------------------------
    # STEP 13: FINAL RESPONSE
    # -------------------------------------------------------

    form_id = f"form_{abs(hash(ocr_text))}"

    return {
        "formSchema": corrected,
        "formId": form_id,
        "title": title,
        "header": header,
        "elements": corrected,
        "stats": stats,
        "instructions": [],
        "declarations": []
    }


In [ ]:
# ============================================================
# EXTRACT PROFILE API
# ============================================================

@app.post("/extract-profile")
async def extract_profile(req: ProfileRequest):

    resume_text = req.resumeText.strip()

    # --------------------------------------------------------
    # STEP 1: VALIDATE RESUME TEXT
    # --------------------------------------------------------

    if not resume_text:

        return {
            "error": "Resume text is empty",
            "profile": {}
        }

    print("=" * 70)
    print("EXTRACTING RESUME PROFILE")
    print("Resume characters:", len(resume_text))
    print("=" * 70)

    # --------------------------------------------------------
    # STEP 2: BUILD QWEN PROMPT
    # --------------------------------------------------------

    prompt = RESUME_PROMPT + resume_text

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    # --------------------------------------------------------
    # STEP 3: APPLY QWEN CHAT TEMPLATE
    # --------------------------------------------------------

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # --------------------------------------------------------
    # STEP 4: TOKENIZE
    # --------------------------------------------------------

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    # --------------------------------------------------------
    # STEP 5: QWEN GENERATION
    # --------------------------------------------------------

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=768,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # --------------------------------------------------------
    # STEP 6: DECODE ONLY GENERATED CONTENT
    # --------------------------------------------------------

    raw = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print("\nRAW PROFILE OUTPUT:")
    print(raw[:5000])

    # --------------------------------------------------------
    # STEP 7: REMOVE MARKDOWN CODE BLOCKS
    # --------------------------------------------------------

    cleaned = re.sub(
        r"```json|```",
        "",
        raw,
        flags=re.IGNORECASE
    ).strip()

    # --------------------------------------------------------
    # STEP 8: EXTRACT JSON OBJECT
    # --------------------------------------------------------

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1:

        print("ERROR: JSON object not found")

        return {
            "error": "Model did not return a JSON object",
            "raw": raw,
            "profile": {}
        }

    cleaned = cleaned[start:end + 1]

    # --------------------------------------------------------
    # STEP 9: PARSE JSON
    # --------------------------------------------------------

    try:

        profile = json.loads(cleaned)

    except json.JSONDecodeError as e:

        print("PROFILE JSON ERROR:", e)

        return {
            "error": "Invalid profile JSON",
            "raw": cleaned,
            "profile": {}
        }

    # --------------------------------------------------------
    # STEP 10: VERIFY OBJECT
    # --------------------------------------------------------

    if not isinstance(profile, dict):

        return {
            "error": "Profile is not a JSON object",
            "raw": cleaned,
            "profile": {}
        }

    # --------------------------------------------------------
    # STEP 11: NORMALIZE PROFILE
    # --------------------------------------------------------
    #
    # Ensures the frontend always receives the same structure.
    # Missing values are replaced with "" or [].
    # --------------------------------------------------------

    normalized_profile = {
        "fullName": profile.get(
            "fullName",
            ""
        ),

        "email": profile.get(
            "email",
            ""
        ),

        "phone": profile.get(
            "phone",
            ""
        ),

        "linkedin": profile.get(
            "linkedin",
            ""
        ),

        "github": profile.get(
            "github",
            ""
        ),

        "portfolio": profile.get(
            "portfolio",
            ""
        ),

        "location": profile.get(
            "location",
            ""
        ),

        "skills": profile.get(
            "skills",
            []
        )
    }

    # --------------------------------------------------------
    # Ensure skills is always a list
    # --------------------------------------------------------

    if not isinstance(
        normalized_profile["skills"],
        list
    ):

        normalized_profile["skills"] = []

    print("\nEXTRACTED PROFILE:")

    print(
        json.dumps(
            normalized_profile,
            indent=2,
            ensure_ascii=False
        )
    )

    print("=" * 70)

    # --------------------------------------------------------
    # STEP 12: FINAL RESPONSE
    # --------------------------------------------------------

    return {
        "profile": normalized_profile
    }


In [ ]:

# ============================================================
# AUTOFILL FORM USING EXTRACTED PROFILE
# ============================================================

@app.post("/autofill")
async def autofill(req: AutofillRequest):

    profile = req.profile
    form_schema = req.formSchema

    # --------------------------------------------------------
    # STEP 1: VALIDATE INPUT
    # --------------------------------------------------------

    if not profile:

        return {
            "error": "Profile is empty",
            "filledSchema": form_schema
        }

    if not form_schema:

        return {
            "error": "Form schema is empty",
            "filledSchema": form_schema
        }

    print("=" * 70)
    print("AUTOFILLING FORM")
    print("Profile fields:", len(profile))
    print("Form fields:", len(form_schema))
    print("=" * 70)

    # --------------------------------------------------------
    # STEP 2: PROCESS EACH FORM FIELD
    # --------------------------------------------------------

    for field in form_schema:

        if not isinstance(field, dict):
            continue

        field_type = field.get("type", "")
        label = field.get("label", "")

        if not label:
            continue

        label = label.lower().strip()

        # ----------------------------------------------------
        # Headings and sections are not fillable
        # ----------------------------------------------------

        if field_type in {
            "heading",
            "section"
        }:
            continue

        value = ""

        # ----------------------------------------------------
        # NAME
        # ----------------------------------------------------

        if (
            ("name" in label and "student" in label)
            or "full name" in label
            or "applicant name" in label
            or label == "name"
        ):
            value = profile.get(
                "fullName",
                ""
            )

        # ----------------------------------------------------
        # EMAIL
        # ----------------------------------------------------

        elif "email" in label:

            value = profile.get(
                "email",
                ""
            )

        # ----------------------------------------------------
        # PHONE / MOBILE
        # ----------------------------------------------------

        elif (
            "mobile" in label
            or "phone" in label
            or "telephone" in label
        ):

            value = profile.get(
                "phone",
                ""
            )

        # ----------------------------------------------------
        # LINKEDIN
        # ----------------------------------------------------

        elif "linkedin" in label:

            value = profile.get(
                "linkedin",
                ""
            )

        # ----------------------------------------------------
        # GITHUB
        # ----------------------------------------------------

        elif "github" in label:

            value = profile.get(
                "github",
                ""
            )

        # ----------------------------------------------------
        # PORTFOLIO
        # ----------------------------------------------------

        elif "portfolio" in label:

            value = profile.get(
                "portfolio",
                ""
            )

        # ----------------------------------------------------
        # LOCATION / ADDRESS
        # ----------------------------------------------------

        elif (
            "address" in label
            or "location" in label
        ):

            value = profile.get(
                "location",
                ""
            )

        # ----------------------------------------------------
        # APPLY VALUE
        # ----------------------------------------------------

        if value:

            field["defaultValue"] = value

            print(
                f"Filled: {field.get('label', '')} "
                f"-> {value}"
            )

    # --------------------------------------------------------
    # STEP 3: RETURN FILLED SCHEMA
    # --------------------------------------------------------

    print("=" * 70)

    return {
        "filledSchema": form_schema
    }


In [ ]:
id="q8m2ka"
# ============================================================
# HEALTH CHECK API
# ============================================================

@app.get("/health")
def health():

    device = "cuda" if torch.cuda.is_available() else "cpu"

    return {
        "status": "ok",
        "model": MODEL_NAME,
        "device": device
    }



In [ ]:

# ============================================================
# TEST /GENERATE-SCHEMA
# ============================================================

test_form_text = """
SREE VENKATA INSTITUTE OF TECHNOLOGY AND SCIENCES

APPLICATION FOR ISSUE OF BONAFIDE CERTIFICATE

APPLICANT DETAILS

Name of Student:

Register Number:

Date of Birth:

Course:

Mobile Number:

Email Address:

PARENT DETAILS

Father's Name:

Mother's Name:
"""

print("=" * 70)
print("TESTING /generate-schema")
print("=" * 70)

result = await generate_schema(
    SchemaRequest(
        extractedText=test_form_text
    )
)

print("\nFINAL API RESPONSE:")
print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False
    )
)

print("=" * 70)


In [ ]:

# ============================================================
# TEST SEMANTIC CORRECTION WITH THE EXACT PROBLEM
# ============================================================

bad_schema = [
    {
        "type": "heading",
        "label": "TRANSFER CERTIFICATE APPLICATION FORM"
    },
    {
        "type": "heading",
        "label": "Page 1 - Student & Academic Information"
    },
    {
        "type": "heading",
        "label": "Application No."
    },
    {
        "type": "heading",
        "label": "Date of Application"
    },
    {
        "type": "heading",
        "label": "Academic Year"
    },
    {
        "type": "heading",
        "label": "Student Name"
    },
    {
        "type": "heading",
        "label": "Date of Birth"
    },
    {
        "type": "heading",
        "label": "Gender"
    },
    {
        "type": "heading",
        "label": "Nationality"
    },
    {
        "type": "heading",
        "label": "Category"
    },
    {
        "type": "heading",
        "label": "Aadhaar / ID Number"
    }
]


print("=" * 70)
print("SEMANTIC CORRECTION TEST")
print("=" * 70)


# ------------------------------------------------------------
# STEP 1: BEFORE
# ------------------------------------------------------------

print("\nBEFORE:")
print(
    json.dumps(
        bad_schema,
        indent=2,
        ensure_ascii=False
    )
)


# ------------------------------------------------------------
# STEP 2: NORMALIZE
# ------------------------------------------------------------

normalized = normalize_form_schema(
    bad_schema
)


print("\nAFTER NORMALIZATION:")
print(
    json.dumps(
        normalized,
        indent=2,
        ensure_ascii=False
    )
)


# ------------------------------------------------------------
# STEP 3: SEMANTIC CORRECTION
# ------------------------------------------------------------

fixed = correct_semantics(
    normalized
)


print("\nAFTER SEMANTIC CORRECTION:")
print(
    json.dumps(
        fixed,
        indent=2,
        ensure_ascii=False
    )
)


# ------------------------------------------------------------
# STEP 4: APPLY OPTIONS SOURCE
# ------------------------------------------------------------

fixed_with_options = [
    apply_options_source(field)
    for field in fixed
]


print("\nAFTER OPTIONS SOURCE:")
print(
    json.dumps(
        fixed_with_options,
        indent=2,
        ensure_ascii=False
    )
)


# ------------------------------------------------------------
# STEP 5: VALIDATION
# ------------------------------------------------------------

validation_errors = validate_form_schema(
    fixed_with_options
)


print("\nVALIDATION:")

if validation_errors:

    for error in validation_errors:
        print("ERROR:", error)

else:

    print("PASSED")


print("=" * 70)



In [ ]:
resume_text = """KAMALI M
+91 8807634655
kamalimahendran0201@gmail.com
LinkedIn: linkedin.com/in/kamali-m
GitHub: github.com/KAMALI216

SKILLS
Java, Python, JavaScript, MySQL, ReactJS, Spring Boot
"""

result = await extract_profile(ProfileRequest(resumeText=resume_text))
print(json.dumps(result, indent=2))

In [ ]:
# ============================================================
# REALISTIC FORM TEST
# ============================================================

test_form_text_2 = """
TRANSFER CERTIFICATE APPLICATION FORM

Page 1 - Student & Academic Information

Application No. ______________________

Date of Application: ________________

Academic Year: ______________________

Student Name: _______________________

Date of Birth: ______________________

Gender: Male [ ] Female [ ] Other [ ]

Nationality: ________________________

Category: General [ ] OBC [ ] SC [ ] ST [ ]

Aadhaar / ID Number: ________________

Father's Name: ______________________

Mother's Name: ______________________

Mobile Number: ______________________

Email Address: ______________________

Permanent Address:
_____________________________________
_____________________________________

DECLARATION

I hereby declare that the information provided is correct.

Signature of Applicant: ______________
"""

result = await generate_schema(
    SchemaRequest(
        extractedText=test_form_text_2
    )
)

print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
id="m8x4qp"
# ============================================================
# TEST AUTOFILL
# ============================================================

test_profile = {
    "fullName": "KAMALI M",
    "email": "kamalimahendran0201@gmail.com",
    "phone": "+91 8807634655"
}


test_schema = [
    {
        "type": "heading",
        "label": "APPLICANT DETAILS"
    },
    {
        "label": "Name of Student",
        "type": "text",
        "required": True
    },
    {
        "label": "Mobile Number",
        "type": "tel",
        "required": True
    },
    {
        "label": "Email Address",
        "type": "email",
        "required": True
    }
]


print("=" * 70)
print("TESTING AUTOFILL")
print("=" * 70)


result = await autofill(
    AutofillRequest(
        profile=test_profile,
        formSchema=test_schema
    )
)


print("\nAUTOFILL RESULT:")

print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False
    )
)


print("=" * 70)


In [ ]:
import os

os.environ["NGROK_AUTH_TOKEN"] = "3IrCznL4f4EW5jWulUT1INoIHxh_6UVxSMe4fcSbDjxbTcuov"

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN")

if not NGROK_AUTH_TOKEN:
    raise RuntimeError(
        "NGROK_AUTH_TOKEN is not set."
    )

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(5001)

print("Public URL:", public_url)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=5001
)

server = uvicorn.Server(config)

await server.serve()

INFO:     Started server process [588]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:5001 (Press CTRL+C to quit)


Public URL: NgrokTunnel: "https://parasitic-shrouded-carrot.ngrok-free.dev" -> "http://localhost:5001"
